# 📈 Stock Price Prediction with Large Language Models (LLM)
### Columbia University — Prompt Engineering & Programming with OpenAI

---

## 🧠 Project Overview

This project demonstrates how **Large Language Models (LLMs)** can perform complex machine learning tasks without traditional coding knowledge.
Instead of writing code manually, I used **Google's Gemini AI** inside Google Colab to:

- 📊 Load and explore Apple (AAPL) historical stock data
- 🔧 Engineer meaningful features for prediction
- 🤖 Build and compare two machine learning models
- 📉 Evaluate model performance on unseen data
- 💼 Discuss real-world trading strategies

**Stock:** Apple Inc. (AAPL)  
**Training Period:** January 1, 2020 — January 1, 2023  
**Test Period:** January 1, 2023 — January 1, 2024

---

## Step 1: Data Exploration 🔍

### What we are doing here:
We download 3 years of Apple stock price history from Yahoo Finance — a free financial data source.
Then we look at key summary statistics to understand the data before building any models.

### Prompt used with Gemini AI:
> *"Load historical stock price data for Apple Inc. (AAPL) from 2020-01-01 to 2023-01-01 using Yahoo Finance and summarize its key statistics."*

In [ ]:
# ============================================================
# STEP 1: Load Apple Stock Data from Yahoo Finance
# ============================================================
# yfinance = a free Python library that downloads stock data
# pandas  = a library for working with data tables

import yfinance as yf
import pandas as pd

# Download 3 years of Apple stock price data
# 'AAPL' is Apple's ticker symbol on the stock market
data = yf.download("AAPL", start="2020-01-01", end="2023-01-01")

# Display summary statistics
# This shows: count, average, min, max, std deviation, and quartiles
print("=== Apple Stock Summary Statistics (2020-2023) ===")
print(data.describe())
print(f"\nTotal trading days loaded: {len(data)}")

### 📋 What the Results Tell Us:

| Statistic | Value | Plain English Meaning |
|---|---|---|
| **Count** | 756 days | Stock markets close on weekends and holidays |
| **Average Close** | ~$127 | Average price Apple stock closed at over 3 years |
| **Min Price** | ~$54 | Lowest price — COVID crash in March 2020 |
| **Max Price** | ~$178 | Highest price — Apple's peak in late 2021 |
| **Std Deviation** | ~$30 | Price bounced around a lot — very volatile stock |
| **Avg Volume** | 112 million | 112 million Apple shares bought/sold every single day |

**Key Insight:** Apple stock went on a massive rollercoaster — crashing during COVID, then soaring to record highs. The standard deviation of ~$30 confirms significant price swings throughout the period.

---

## Step 2: Feature Engineering 🔧

### What is Feature Engineering?
Feature Engineering means **creating new useful columns from existing data** to help the model find better patterns.
Think of it like giving the model extra clues beyond just the raw price.

For example: Instead of just knowing today's price, we also tell the model:
- What was the average price over the last 5 days?
- Is the stock gaining or losing momentum?
- How volatile has it been recently?

### Features we created:

| Feature | What It Means |
|---|---|
| **SMA 5, 10, 20, 50, 200** | Average closing price over last 5/10/20/50/200 days |
| **RSI** | Relative Strength Index — is the stock overbought or oversold? |
| **Williams %R** | Momentum indicator — price relative to recent high/low range |
| **Bollinger Bands** | Price volatility range — upper, middle, and lower price bands |
| **ADX** | Average Directional Index — strength of the current trend |
| **OBV** | On-Balance Volume — tracks buying vs selling pressure |
| **Ichimoku** | Japanese trend indicator showing support/resistance levels |
| **Lagged Returns** | Price returns from 5 and 20 days ago — captures momentum |
| **Time Features** | Quarter, Day of Year, Week of Year — captures seasonal patterns |

### Prompt used with Gemini AI:
> *"Using the AAPL data already loaded, add moving averages (SMA 5, 10, 20, 50, 200), RSI, Williams %R, Bollinger Bands, ADX, OBV, Ichimoku indicators, lagged returns, and time-based features."*

In [ ]:
# ============================================================
# STEP 2A: Install and import the Technical Analysis library
# ============================================================
# 'ta' is a Python library that calculates stock technical indicators

!pip install ta --quiet
import ta
import numpy as np

print("Libraries installed and imported successfully!")

In [ ]:
# ============================================================
# STEP 2B: Add Simple Moving Averages (SMA)
# ============================================================
# SMA = average closing price over the last N days
# Example: SMA_5 = average of last 5 closing prices
# Why useful: Smooths out daily noise to show the overall trend

close = data['Close'].squeeze()  # Extract closing price as a simple series

data['SMA_5']   = close.rolling(window=5).mean()    # 5-day average
data['SMA_10']  = close.rolling(window=10).mean()   # 10-day average
data['SMA_20']  = close.rolling(window=20).mean()   # 20-day average
data['SMA_50']  = close.rolling(window=50).mean()   # 50-day average
data['SMA_200'] = close.rolling(window=200).mean()  # 200-day average (long term trend)

print("Moving Averages added: SMA_5, SMA_10, SMA_20, SMA_50, SMA_200")

In [ ]:
# ============================================================
# STEP 2C: Add RSI (Relative Strength Index)
# ============================================================
# RSI measures if a stock is overbought (too high, likely to fall)
# or oversold (too low, likely to rise)
# RSI > 70 = overbought (sell signal)
# RSI < 30 = oversold (buy signal)
# Standard window = 14 days

data['RSI'] = ta.momentum.RSIIndicator(close, window=14).rsi()

print("RSI (Relative Strength Index) added")

In [ ]:
# ============================================================
# STEP 2D: Add Williams %R
# ============================================================
# Williams %R is similar to RSI — another momentum indicator
# It shows where the current price is relative to the recent high/low range
# Range: -100 to 0
# Near 0 = overbought, Near -100 = oversold

high  = data['High'].squeeze()
low   = data['Low'].squeeze()

data['Williams_%R'] = ta.momentum.WilliamsRIndicator(
    high=high, low=low, close=close
).williams_r()

print("Williams %R added")

In [ ]:
# ============================================================
# STEP 2E: Add Bollinger Bands
# ============================================================
# Bollinger Bands create a price 'envelope' around the moving average
# Upper Band = moving average + 2 standard deviations (upper limit)
# Lower Band = moving average - 2 standard deviations (lower limit)
# When price touches upper band = potentially overbought
# When price touches lower band = potentially oversold
# BB_Width = how wide the bands are (wider = more volatile market)

bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
data['BB_Upper']  = bb.bollinger_hband()   # Upper band
data['BB_Middle'] = bb.bollinger_mavg()    # Middle band (20-day SMA)
data['BB_Lower']  = bb.bollinger_lband()   # Lower band
data['BB_Width']  = data['BB_Upper'] - data['BB_Lower']  # Band width

print("Bollinger Bands added: BB_Upper, BB_Middle, BB_Lower, BB_Width")

In [ ]:
# ============================================================
# STEP 2F: Add ADX (Average Directional Index)
# ============================================================
# ADX measures the STRENGTH of a trend (not its direction)
# ADX > 25 = strong trend (either up or down)
# ADX < 20 = weak trend or sideways market
# ADX_pos = positive directional indicator (upward pressure)
# ADX_neg = negative directional indicator (downward pressure)

adx = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14)
data['ADX']     = adx.adx()      # Overall trend strength
data['ADX_pos'] = adx.adx_pos()  # Upward movement strength
data['ADX_neg'] = adx.adx_neg()  # Downward movement strength

print("ADX (Average Directional Index) added")

In [ ]:
# ============================================================
# STEP 2G: Add OBV (On-Balance Volume)
# ============================================================
# OBV tracks buying vs selling pressure using volume
# If stock closes UP today: add today's volume to OBV
# If stock closes DOWN today: subtract today's volume from OBV
# Rising OBV = more buyers than sellers = bullish signal
# Falling OBV = more sellers than buyers = bearish signal

volume = data['Volume'].squeeze()
data['OBV'] = ta.volume.OnBalanceVolumeIndicator(
    close=close, volume=volume
).on_balance_volume()

print("OBV (On-Balance Volume) added")

In [ ]:
# ============================================================
# STEP 2H: Add Time-Based Features
# ============================================================
# Stock prices often show seasonal patterns
# For example: stocks often perform differently in Q4 vs Q1
# Adding time features helps the model detect these patterns

data['Month']        = data.index.month           # Month (1-12)
data['Quarter']      = data.index.quarter         # Quarter (1-4)
data['Day_of_Year']  = data.index.day_of_year     # Day of year (1-365)
data['Week_of_Year'] = data.index.isocalendar().week.astype(int)  # Week number

print("Time-based features added: Month, Quarter, Day_of_Year, Week_of_Year")

In [ ]:
# ============================================================
# STEP 2I: Add Lagged Returns
# ============================================================
# Lagged return = how much the price changed X days ago
# This captures momentum — if the stock was going up 5 days ago,
# does it tend to continue going up today?
# pct_change() calculates the percentage change from previous row

data['Lagged_Return_5D']  = close.pct_change(5)   # Return over last 5 days
data['Lagged_Return_20D'] = close.pct_change(20)  # Return over last 20 days

print("Lagged returns added: Lagged_Return_5D, Lagged_Return_20D")

In [ ]:
# ============================================================
# STEP 2J: Preview the final feature-engineered dataset
# ============================================================

print(f"Total features created: {len(data.columns)}")
print(f"Total rows (trading days): {len(data)}")
print("\nAll feature columns:")
print(list(data.columns))
print("\nLast 5 rows of data:")
print(data.tail())

---
## Step 3: Model Building 🤖

We built **two different machine learning models** and compared their performance.

### Model 3a — Linear Regression (Predict Exact Price)
Tries to predict **the exact closing price** for tomorrow by finding a straight-line pattern in past data.

### Model 3b — Random Forest Classification (Predict UP or DOWN)
Instead of predicting the exact price, asks a simpler question: **Will Apple stock go UP or DOWN tomorrow?**
Uses 100 different decision trees that each vote on the answer — majority vote wins.

### What is Train/Test Split?
We split our data **70% for training** and **30% for testing**:
- **Training data (70%)**: The model learns patterns from this data
- **Testing data (30%)**: We test the model on data it has NEVER seen before

This is important because we want to know if the model works on NEW data — not just the data it already learned from!

### Prompt used with Gemini AI:
> *"Train a Linear Regression model to predict tomorrow's closing price and a Random Forest model to predict price direction (up/down). Use 70/30 train-test split. Report MSE, R² score, accuracy, and feature importance."*

In [ ]:
# ============================================================
# STEP 3A: Prepare data for modeling
# ============================================================
# Before building models we need to:
# 1. Flatten the multi-level column names (a Pandas formatting issue)
# 2. Create the TARGET variable — what we want to predict
# 3. Remove rows with missing values (NaN) caused by rolling calculations
# 4. Split into features (X) and target (y)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report

# Flatten column names (removes the multi-level 'Price/AAPL' header)
data.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col 
                for col in data.columns.values]

# Create target for Linear Regression: tomorrow's closing price
# shift(-1) moves the close price UP by 1 row so each row shows TOMORROW's price
data['Target_Price'] = data['Close_AAPL'].shift(-1)

# Create target for Random Forest: will price go UP (1) or DOWN (0) tomorrow?
data['Target_Direction'] = (data['Close_AAPL'].shift(-1) > data['Close_AAPL']).astype(int)

# Drop rows with missing values (first rows missing due to rolling windows)
data_clean = data.dropna()

print(f"Rows after cleaning: {len(data_clean)} (removed {len(data) - len(data_clean)} rows with missing values)")
print(f"Price goes UP tomorrow: {data_clean['Target_Direction'].sum()} days")
print(f"Price goes DOWN tomorrow: {(data_clean['Target_Direction'] == 0).sum()} days")

In [ ]:
# ============================================================
# STEP 3B: Define features and split data 70/30
# ============================================================
# Features (X) = all the columns the model uses to learn
# Target (y)   = the answer we want to predict

# Select feature columns (exclude target columns and non-numeric)
feature_cols = ['Close_AAPL', 'High_AAPL', 'Low_AAPL', 'Open_AAPL', 'Volume_AAPL',
                'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50', 'SMA_200',
                'RSI', 'Williams_%R', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'BB_Width',
                'ADX', 'ADX_pos', 'ADX_neg', 'OBV',
                'Month', 'Quarter', 'Day_of_Year', 'Week_of_Year',
                'Lagged_Return_5D', 'Lagged_Return_20D']

# Only use columns that exist in the cleaned data
feature_cols = [col for col in feature_cols if col in data_clean.columns]

X = data_clean[feature_cols]
y_price     = data_clean['Target_Price']      # For Linear Regression
y_direction = data_clean['Target_Direction']  # For Random Forest

# Split data: 70% training, 30% testing
# shuffle=False is CRITICAL for time series — we must not mix past and future data!
X_train, X_test, y_price_train, y_price_test = train_test_split(
    X, y_price, test_size=0.3, shuffle=False
)
_, _, y_dir_train, y_dir_test = train_test_split(
    X, y_direction, test_size=0.3, shuffle=False
)

print(f"Training samples: {len(X_train)} rows (70%)")
print(f"Testing samples:  {len(X_test)} rows (30%)")

In [ ]:
# ============================================================
# STEP 3C: Linear Regression — Predict Exact Stock Price
# ============================================================
# Linear Regression tries to find a straight-line relationship
# between our features and tomorrow's price
#
# Performance Metrics:
# MSE (Mean Squared Error) = average squared error of predictions
#   → Lower is better. If MSE = 859, predictions are off by ~$29 on average
# R² Score = how well the model explains price variation
#   → 1.0 = perfect, 0 = no better than guessing average, negative = very bad

lr_model = LinearRegression()
lr_model.fit(X_train, y_price_train)  # Train the model

y_price_pred = lr_model.predict(X_test)  # Make predictions on test data

mse = mean_squared_error(y_price_test, y_price_pred)
r2  = r2_score(y_price_test, y_price_pred)

print("=== Linear Regression Results ===")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score:                 {r2:.4f}")
print()
print("📌 What this means:")
print(f"  → Predictions are off by approximately ${mse**0.5:.2f} per day on average")
if r2 < 0:
    print("  → Negative R² means the model is WORSE than simply guessing the average price every day!")
    print("  → Stock prices are too complex for a simple straight-line model")

In [ ]:
# ============================================================
# STEP 3D: Random Forest — Predict UP or DOWN Direction
# ============================================================
# Random Forest creates 100 decision trees, each learning different
# patterns from the data. Then all 100 trees VOTE on the answer.
# Majority vote wins — like asking 100 friends for advice!
#
# This is a CLASSIFICATION model:
#   → Predicts 1 (price will go UP tomorrow)
#   → Predicts 0 (price will go DOWN tomorrow)
#
# n_estimators=100 means we use 100 decision trees
# random_state=42  ensures results are reproducible

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_dir_train)  # Train the model

y_dir_pred = rf_model.predict(X_test)  # Make predictions on test data

accuracy = accuracy_score(y_dir_test, y_dir_pred)

print("=== Random Forest Classification Results ===")
print(f"Accuracy: {accuracy * 100:.2f}%")
print()
print(f"📌 What this means:")
print(f"  → The model correctly predicted UP or DOWN on {accuracy * 100:.1f}% of days")
print()
print("Detailed Classification Report:")
print(classification_report(y_dir_test, y_dir_pred, 
                             target_names=['Price DOWN (0)', 'Price UP (1)']))

In [ ]:
# ============================================================
# STEP 3E: Feature Importance — What did the model rely on most?
# ============================================================
# Random Forest can tell us which features were most useful
# for making predictions. Higher importance = more influential.

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("=== Top 10 Most Important Features ===")
print(importance_df.head(10).to_string(index=False))
print()
print("📌 What this means:")
print("  → Features with higher importance had more influence on the model's predictions")
print("  → The closing price and short-term moving averages are typically most predictive")

---
## Step 4: Model Evaluation on New Data 📊

### What we are doing here:
Now we test our trained Random Forest model on **completely new data it has never seen** — Apple stock from 2023 to 2024.

This is the **real test** of how well our model actually works in the real world.

### Why accuracy might drop:
A drop in accuracy from training to new data is called **overfitting** — the model learned the patterns of 2020–2023 very well but struggles to generalize to a new time period. This is a common challenge in machine learning.

### Prompt used with Gemini AI:
> *"Download Apple stock data from 2023-01-01 to 2024-01-01. Apply the same features as before and use the trained Random Forest model to predict price direction. Report accuracy and classification metrics."*

In [ ]:
# ============================================================
# STEP 4A: Download new 2023-2024 Apple stock data
# ============================================================
# We download one year of NEW data the model has never seen
# This simulates how the model would perform in real life

data_new = yf.download("AAPL", start="2023-01-01", end="2024-01-01", progress=False)

# Flatten column names to match training data format
data_new.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col 
                    for col in data_new.columns.values]

print(f"New data downloaded: {len(data_new)} trading days (2023)")
print(data_new.head())

In [ ]:
# ============================================================
# STEP 4B: Apply the same features to new data
# ============================================================
# We must create the exact same features on the new data
# so the model can understand and make predictions

close_new  = data_new['Close_AAPL']
high_new   = data_new['High_AAPL']
low_new    = data_new['Low_AAPL']
volume_new = data_new['Volume_AAPL']

# Add all the same features as Step 2
data_new['SMA_5']   = close_new.rolling(5).mean()
data_new['SMA_10']  = close_new.rolling(10).mean()
data_new['SMA_20']  = close_new.rolling(20).mean()
data_new['SMA_50']  = close_new.rolling(50).mean()
data_new['SMA_200'] = close_new.rolling(200).mean()
data_new['RSI']     = ta.momentum.RSIIndicator(close_new, window=14).rsi()
data_new['Williams_%R'] = ta.momentum.WilliamsRIndicator(high=high_new, low=low_new, close=close_new).williams_r()

bb_new = ta.volatility.BollingerBands(close_new, window=20)
data_new['BB_Upper']  = bb_new.bollinger_hband()
data_new['BB_Middle'] = bb_new.bollinger_mavg()
data_new['BB_Lower']  = bb_new.bollinger_lband()
data_new['BB_Width']  = data_new['BB_Upper'] - data_new['BB_Lower']

adx_new = ta.trend.ADXIndicator(high=high_new, low=low_new, close=close_new)
data_new['ADX']     = adx_new.adx()
data_new['ADX_pos'] = adx_new.adx_pos()
data_new['ADX_neg'] = adx_new.adx_neg()
data_new['OBV']     = ta.volume.OnBalanceVolumeIndicator(close=close_new, volume=volume_new).on_balance_volume()

data_new['Month']        = data_new.index.month
data_new['Quarter']      = data_new.index.quarter
data_new['Day_of_Year']  = data_new.index.day_of_year
data_new['Week_of_Year'] = data_new.index.isocalendar().week.astype(int)
data_new['Lagged_Return_5D']  = close_new.pct_change(5)
data_new['Lagged_Return_20D'] = close_new.pct_change(20)

# Create target: did price go up the next day?
data_new['Target_Direction'] = (close_new.shift(-1) > close_new).astype(int)

# Remove missing values
data_new_clean = data_new.dropna()
print(f"New data ready for prediction: {len(data_new_clean)} rows")

In [ ]:
# ============================================================
# STEP 4C: Make predictions on new 2023-2024 data
# ============================================================
# Use only the features the model was trained on
# Then compare predictions to actual outcomes

available_features = [col for col in feature_cols if col in data_new_clean.columns]
X_new = data_new_clean[available_features]
y_new = data_new_clean['Target_Direction']

# Make predictions using the already-trained Random Forest model
y_new_pred = rf_model.predict(X_new)

new_accuracy = accuracy_score(y_new, y_new_pred)

print("=== Model Performance on NEW 2023-2024 Data ===")
print(f"Accuracy: {new_accuracy * 100:.2f}%")
print()
print("Detailed Classification Report:")
print(classification_report(y_new, y_new_pred,
                             target_names=['Price DOWN (0)', 'Price UP (1)']))
print()
print("📌 Comparison Summary:")
print(f"  Training period accuracy (2020-2023): {accuracy * 100:.2f}%")
print(f"  New data accuracy       (2023-2024): {new_accuracy * 100:.2f}%")
print()
if new_accuracy < accuracy:
    print("  → Accuracy dropped on new data — this is called 'overfitting'")
    print("  → The model learned 2020-2023 patterns but struggles with new market conditions")

---
## Step 5: Trading Strategy Discussion 💼

### What we are doing here:
Based on our model's performance, we discuss potential trading strategies and their limitations.
This is the critical thinking part — evaluating whether the model is actually useful for real decisions.

### Prompt used with Gemini AI:
> *"Based on the Random Forest model performance, what trading strategies could I build? What are the limitations of using this model for real financial decisions?"*

---

### 🟢 Strategy 1: Conservative Strategy
- **When to act:** Only buy when the model predicts a price increase with HIGH confidence
- **Logic:** Be very selective — only trade on the model's strongest signals
- **Risk management:** Stay out of the market when the model is uncertain
- **Best for:** Risk-averse investors who prefer fewer but more reliable trades

---

### 🔴 Strategy 2: Contrarian Strategy
- **When to act:** Stay out of the market when the model predicts NO price increase
- **Logic:** The model is better at identifying 'no increase' days, so trust those predictions
- **Risk management:** Requires careful monitoring to avoid false signals
- **Best for:** Investors who want to avoid losses rather than chase gains

---

### 🟡 Strategy 3: Hybrid Strategy
- **When to act:** Set confidence thresholds — only buy when probability is very high, only sell when model is very confident
- **Logic:** Combine the best of both strategies with flexible rules
- **Risk management:** Adjust thresholds based on recent model performance and market conditions
- **Best for:** Active investors comfortable with monitoring and adjusting strategy

---

### ⚠️ Important Limitations

| Limitation | Why It Matters |
|---|---|
| **60-71% accuracy is not perfect** | The model is wrong 29-40% of the time — real money is at risk |
| **Overfitting** | Model learned past patterns that may not repeat in the future |
| **Missing data** | Model doesn't see news, earnings reports, or world events |
| **Market changes** | Stock market behavior changes — what worked in 2020-2023 may not work in 2024+ |
| **Transaction costs** | Real trading has fees that eat into profits from small gains |

> **⚠️ Disclaimer:** This model is for educational purposes only. It should NOT be used as the sole basis for real financial trading decisions. Always consult a qualified financial advisor before investing.

---
## 📌 Key Takeaways & Conclusions

### What I Learned:

| Concept | What I Learned |
|---|---|
| **Prompt Engineering** | Clear, specific prompts produce better AI-generated code |
| **Data Exploration** | Always understand your data before building models |
| **Feature Engineering** | Better features = better models. Raw data alone is rarely enough |
| **Model Selection** | Classification (UP/DOWN) beat regression (exact price) for this problem |
| **Model Evaluation** | Always test on NEW unseen data — training accuracy alone is misleading |
| **Critical Thinking** | A 60% accurate model is interesting, but not reliable enough for real trading |
| **LLM Collaboration** | LLMs can write and refine code, but human judgment is essential to guide and validate results |

### Results Summary:

| Model | Task | Result |
|---|---|---|
| **Linear Regression** | Predict exact price | ❌ Failed (R² = negative) |
| **Random Forest (training)** | Predict UP/DOWN | ✅ 71.82% accuracy |
| **Random Forest (new data)** | Predict UP/DOWN | ⚠️ 60.85% accuracy (dropped due to overfitting) |

### Final Conclusion:
This project demonstrates that **predicting stock price direction is more achievable than predicting exact prices**, and that machine learning models trained on historical data can find real patterns. However, the accuracy drop from 71% to 60% on new data reminds us that **stock markets are complex, ever-changing systems** — and no model should be solely relied upon for real investment decisions.

---
*Columbia University — Prompt Engineering & Programming with OpenAI*